# 02 — Combine Tracking EDA

**Purpose**: Explore the 10 Hz Combine tracking data in depth.
Understand speed, acceleration, direction, and movement trajectories by drill type.
Extract the movement feature library.

**Output**:
- `outputs/figures/` — all visualization figures
- `data/processed/combine_features.parquet` — per-attempt movement features
- `data/processed/player_master.parquet` — master player table

> **Reminder**: This is exploratory. Conclusions are hypotheses, not facts yet.

In [ ]:
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ensure_dirs, FIGURES_DIR, DATA_PROCESSED, COMBINE_FEATURES_PARQUET
from src.data_loading import load_players, load_combine_results, load_combine_tracking
from src.player_mapping import assign_position_groups, build_master_player_table
from src.combine_features import (
    extract_features_from_tracking, aggregate_player_features, sort_attempt
)
from src.visualization import (
    plot_drill_player_counts, plot_drill_duration_distribution,
    plot_speed_distribution_by_drill, plot_acceleration_distribution,
    plot_movement_trajectory, plot_trajectories_comparison,
    plot_position_group_profiles, plot_missing_data_heatmap,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(name)s | %(message)s')
ensure_dirs()
print('Setup complete. Ready to explore combine tracking data.')

## 1. Load Data

In [ ]:
players = load_players()
combine_results = load_combine_results(cols=None)  # all columns
tracking = load_combine_tracking()

# Assign position groups
players = assign_position_groups(players)

print(f'Players: {len(players):,}')
print(f'Combine tracking rows: {len(tracking):,}')
print(f'Position groups: {players["position_group"].value_counts().to_dict()}')

## 2. Drill Overview

### 2a. Player counts per drill

In [ ]:
# How many players participated in each drill?
fig = plot_drill_player_counts(tracking)
plt.show()

if 'drill_type' in tracking.columns:
    drill_summary = tracking.groupby('drill_type').agg(
        n_players=('nfl_id', 'nunique'),
        n_rows=('nfl_id', 'count'),
    ).sort_values('n_players', ascending=False)
    display(drill_summary)

### 2b. Drill duration distributions

In [ ]:
fig = plot_drill_duration_distribution(tracking)
plt.show()

## 3. Speed Distributions

In [ ]:
if 's' in tracking.columns:
    fig = plot_speed_distribution_by_drill(tracking, max_speed_only=True)
    plt.show()

    # Overall speed distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(tracking['s'].dropna(), bins=60, color='#003f7f', edgecolor='white', linewidth=0.5)
    axes[0].set_title('Instantaneous Speed Distribution (All Drills)', fontweight='bold')
    axes[0].set_xlabel('Speed (yards/s)')
    axes[0].set_ylabel('Frame count')

    # Max speed per attempt
    keys = [c for c in ['nfl_id', 'drill_type', 'attempt'] if c in tracking.columns]
    max_speed_per_attempt = tracking.groupby(keys)['s'].max().reset_index(name='max_speed')
    axes[1].hist(max_speed_per_attempt['max_speed'].dropna(), bins=50, color='#cc0000', edgecolor='white', linewidth=0.5)
    axes[1].set_title('Peak Speed per Attempt', fontweight='bold')
    axes[1].set_xlabel('Peak Speed (yards/s)')
    axes[1].set_ylabel('Number of attempts')

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'speed_overall_dist.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Speed column (s) not found in tracking data.')

## 4. Acceleration Distributions

**Research question**: Do high-acceleration athletes at the Combine become better NFL performers?
First we need to understand the distribution.

In [ ]:
if 'a' in tracking.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    acc_vals = tracking['a'].dropna()
    axes[0].hist(acc_vals, bins=80, color='#003f7f', edgecolor='white', linewidth=0.3)
    axes[0].set_title('Instantaneous Acceleration Distribution', fontweight='bold')
    axes[0].set_xlabel('Acceleration (yards/s²)')
    axes[0].set_ylabel('Frame count')
    axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero')
    axes[0].legend()

    # Positive vs negative acceleration
    pos_acc = acc_vals[acc_vals > 0]
    neg_acc = acc_vals[acc_vals < 0]
    axes[1].hist(pos_acc, bins=50, color='#228B22', alpha=0.7, edgecolor='white', label='Acceleration', linewidth=0.3)
    axes[1].hist(neg_acc.abs(), bins=50, color='#cc0000', alpha=0.7, edgecolor='white', label='Deceleration (|a|)', linewidth=0.3)
    axes[1].set_title('Acceleration vs Deceleration Magnitude', fontweight='bold')
    axes[1].set_xlabel('Magnitude (yards/s²)')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'acceleration_overall_dist.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Mean acceleration: {acc_vals.mean():.3f} yd/s²')
    print(f'Mean positive acceleration: {pos_acc.mean():.3f} yd/s²')
    print(f'Mean deceleration: {neg_acc.mean():.3f} yd/s²')
    print(f'Max acceleration: {acc_vals.max():.3f} yd/s²')
    print(f'Max deceleration: {acc_vals.min():.3f} yd/s²')
else:
    print('Acceleration column (a) not found — will derive from speed.')

## 5. Direction Distribution

Direction should vary widely in multi-directional drills (3-cone, shuttle)
and be nearly uniform in straight-line drills (40-yard dash).

In [ ]:
if 'dir' in tracking.columns and 'drill_type' in tracking.columns:
    drill_types = tracking['drill_type'].dropna().unique()
    n_drills = len(drill_types)
    fig, axes = plt.subplots(1, min(n_drills, 4), figsize=(4 * min(n_drills, 4), 4),
                             subplot_kw=dict(polar=True))
    if n_drills == 1:
        axes = [axes]

    for ax, drill in zip(axes, drill_types[:4]):
        subset = tracking[tracking['drill_type'] == drill]['dir'].dropna()
        if len(subset) == 0:
            continue
        # Polar histogram
        bins = np.linspace(0, 2 * np.pi, 37)
        radians = np.deg2rad(subset)
        counts, _ = np.histogram(radians, bins=bins)
        width = (2 * np.pi) / 36
        bars = ax.bar(bins[:-1], counts, width=width, color='#003f7f', alpha=0.7, edgecolor='white')
        ax.set_title(drill, fontweight='bold', pad=10)
        ax.set_theta_zero_location('N')
        ax.set_theta_direction(-1)

    fig.suptitle('Direction Distribution by Drill Type', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'direction_polar_by_drill.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('dir or drill_type column missing.')

## 6. Example Movement Trajectories

Visualize 3-5 example trajectories for different drills, colored by speed.
This gives intuition for what the tracking data looks like and helps us understand
what features are feasible to extract.

In [ ]:
if 'drill_type' in tracking.columns and 'x' in tracking.columns:
    drill_types = tracking['drill_type'].dropna().unique()

    for drill in drill_types[:3]:  # Plot 3 drill types
        drill_data = tracking[tracking['drill_type'] == drill]
        if len(drill_data) == 0:
            continue

        # Pick first 3 unique players with enough frames
        group_keys = [c for c in ['nfl_id', 'attempt'] if c in drill_data.columns]
        sample_attempts = (
            drill_data.groupby(group_keys)
            .size()
            .reset_index(name='n_frames')
            .query('n_frames >= 10')
            .head(3)
        )

        if len(sample_attempts) == 0:
            continue

        fig, axes = plt.subplots(1, len(sample_attempts), figsize=(6 * len(sample_attempts), 6))
        if len(sample_attempts) == 1:
            axes = [axes]

        for ax, (_, row) in zip(axes, sample_attempts.iterrows()):
            mask = drill_data['nfl_id'] == row['nfl_id']
            if 'attempt' in drill_data.columns:
                mask &= drill_data['attempt'] == row['attempt']
            attempt_df = sort_attempt(drill_data[mask])
            if len(attempt_df) < 3:
                continue
            title = f'nfl_id={row["nfl_id"]}'
            plot_movement_trajectory(attempt_df, color_by='s', title=title, ax=ax)

        fig.suptitle(f'Movement Trajectories — {drill}', fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        fname = f'trajectory_examples_{drill.replace(" ", "_").replace("/", "_")}.png'
        plt.savefig(FIGURES_DIR / fname, dpi=150, bbox_inches='tight')
        plt.show()
else:
    print('Cannot plot trajectories: missing x/y or drill_type columns.')

## 7. Extract Movement Features

This is the core computation. For every (player, drill, attempt) triple,
compute all movement features.

**This may take a few minutes for large tracking files.**

In [ ]:
print('Extracting movement features from combine tracking...')
print(f'Total attempts to process: check above')

features_df = extract_features_from_tracking(tracking, verbose=True)

print(f'\nFeatures extracted: {features_df.shape}')
print(f'Columns: {features_df.columns.tolist()}')
display(features_df.head())

# Check for NaN-heavy columns
nan_pct = (features_df.isnull().sum() / len(features_df) * 100).sort_values(ascending=False)
print('\nFeature columns with >10% NaN:')
display(nan_pct[nan_pct > 10].to_frame('nan_pct'))

In [ ]:
# Save to Parquet for downstream notebooks
features_df.to_parquet(COMBINE_FEATURES_PARQUET, index=False)
print(f'Saved: {COMBINE_FEATURES_PARQUET}')

# Also create per-player aggregated features (best attempt per drill)
player_features = aggregate_player_features(features_df, agg_strategy='best_attempt')
print(f'Player-level features: {player_features.shape}')
display(player_features.head())

## 8. Feature Distributions by Drill

In [ ]:
# Visualize key feature distributions
key_features = [
    'max_speed', 'first_step_acceleration', 'total_direction_change',
    'movement_efficiency', 'best_peak_reaccel', 'time_to_75pct_peak',
]
key_features = [f for f in key_features if f in features_df.columns]

if 'drill_type' in features_df.columns:
    for feat in key_features:
        valid = features_df[['drill_type', feat]].dropna()
        if len(valid) < 10:
            continue
        order = valid.groupby('drill_type')[feat].median().sort_values(ascending=False).index
        fig, ax = plt.subplots(figsize=(10, 5))
        sns.boxplot(data=valid, x='drill_type', y=feat, order=order, color='#4da6ff', ax=ax)
        ax.set_title(f'{feat} by Drill Type', fontweight='bold')
        ax.set_xlabel('Drill Type')
        ax.set_ylabel(feat)
        ax.tick_params(axis='x', rotation=30)
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / f'feature_{feat}_by_drill.png', dpi=150, bbox_inches='tight')
        plt.show()

## 9. Compare Tracking Features vs Traditional Combine Metrics

**Research question**: Which tracking features provide information not captured by traditional metrics?

e.g., Do two players with the same 40-yard time have different acceleration profiles?

In [ ]:
from src.statistics import correlation_matrix, build_correlation_summary
from src.visualization import plot_correlation_heatmap, plot_combine_vs_nfl
from src.config import TRADITIONAL_COMBINE_COLS

# Build master player table
from src.player_mapping import build_master_player_table
master = build_master_player_table(players, combine_results=combine_results)

# Join with player-level tracking features (best attempt per drill)
# Pivot to wide format: one row per player
if 'drill_type' in player_features.columns:
    # Pivot: drill_type becomes column prefixes
    feat_cols = [c for c in player_features.columns if c not in ['nfl_id', 'drill_type', 'attempt']]
    player_feat_wide = player_features.pivot_table(
        index='nfl_id', columns='drill_type', values=feat_cols, aggfunc='first'
    )
    player_feat_wide.columns = ['_'.join(map(str, c)).strip() for c in player_feat_wide.columns]
    player_feat_wide = player_feat_wide.reset_index()
    
    master_features = master.merge(player_feat_wide, on='nfl_id', how='left')
    print(f'Master + features: {master_features.shape}')
else:
    master_features = master.merge(player_features.drop(columns=['attempt'], errors='ignore'), on='nfl_id', how='left')
    print(f'Master + features (no drill pivot): {master_features.shape}')

In [ ]:
# Correlation: tracking features vs traditional combine metrics
traditional_metrics = [
    c for c in ['ten_yd_split', 'forty', 'vertical', 'broad_jump', 
                 'three_cone', 'short_shuttle', 'ngs_athleticism_score']
    if c in master_features.columns
]

tracking_feats = [
    c for c in master_features.columns
    if any(c.startswith(f) for f in [
        'max_speed', 'first_step_acceleration', 'total_direction_change',
        'movement_efficiency', 'best_peak_reaccel', 'mean_speed',
        'max_acceleration', 'time_to_75pct_peak',
    ])
]

print(f'Traditional metrics available: {traditional_metrics}')
print(f'Tracking features available: {len(tracking_feats)}')

if traditional_metrics and tracking_feats:
    corr_mat = correlation_matrix(
        master_features,
        x_cols=tracking_feats[:15],  # limit for readability
        y_cols=traditional_metrics,
        method='spearman',
    )
    print('\nTracking vs Traditional — Spearman correlation matrix:')
    display(corr_mat)

    if not corr_mat.empty:
        plot_correlation_heatmap(
            corr_mat,
            title='Tracking Features vs Traditional Combine Metrics (Spearman ρ)',
            filename='corr_tracking_vs_traditional.png',
        )
        plt.show()
else:
    print('NOT YET COMPUTED — insufficient columns available for this analysis.')

## 10. Position Group Movement Profiles

Do different position groups move fundamentally differently at the Combine?
This helps us understand which positions are most worth investigating.

In [ ]:
# Join position info onto player_features
pos_map = players[['nfl_id', 'position_group']].drop_duplicates()
feat_with_pos = player_features.merge(pos_map, on='nfl_id', how='left')

profile_feats = [f for f in [
    'max_speed', 'first_step_acceleration', 'total_direction_change',
    'movement_efficiency', 'best_peak_reaccel', 'time_to_75pct_peak',
] if f in feat_with_pos.columns]

if 'position_group' in feat_with_pos.columns and profile_feats:
    print('Sample sizes per position group:')
    display(feat_with_pos['position_group'].value_counts().to_frame())

    plot_position_group_profiles(
        feat_with_pos,
        feature_cols=profile_feats,
        position_col='position_group',
    )
    plt.show()

    # Summary stats by position group
    from src.statistics import compare_groups
    for feat in profile_feats:
        print(f'\n--- {feat} by position group ---')
        display(compare_groups(feat_with_pos, 'position_group', feat))
else:
    print('Position group or feature columns not available.')